# Operating-System Cryptographic Random-Number Analysis

This notebook evaluates Linux `/dev/urandom`, OpenSSL `rand`, and Windows CNG/.NET random generation. Each source is sampled for at least 1 MiB. Passing distribution tests does not establish cryptographic security.

## Linux `/dev/urandom`
Linux maintains a kernel cryptographic random-number generator. The `/dev/urandom` interface returns output without the historical blocking behavior of `/dev/random`, while mixing hardware and system entropy into its internal state. It is suitable for cryptographic key and nonce generation in normal modern Linux use. Initial boot and virtual-machine entropy quality should still be considered by system administrators.

## Windows CNG
Windows provides cryptographic randomness through CNG APIs such as `BCryptGenRandom`. .NET `RandomNumberGenerator.GetBytes` uses the platform cryptographic implementation, which on Windows is backed by CNG. Windows also manages kernel entropy pools, OS entropy sources, hardware RNGs where available, and secure RNG state.

## Statistical methodology
The analysis measures byte frequency, chi-square deviation from 256 uniformly expected byte classes, one-bit frequency, runs, Shannon entropy, lag-1 serial correlation, and 16-byte block uniqueness. These tests detect gross bias and repeating behavior, but not cryptographic predictability.

In [ ]:
import sys
from pathlib import Path
workspace = Path.cwd()
if workspace.name == 'notebooks':
    workspace = workspace.parent.parent
sys.path.insert(0, str(workspace))
from modern.os_random_analysis.src.os_random_analysis import run_collectors, format_report
size = 1024 * 1024
output_dir = workspace / 'modern' / 'os_random_analysis' / 'outputs'
source_results = run_collectors(size, output_dir)
print(format_report(source_results))


## Comparison
All three mechanisms should produce distributions close to uniform at the byte and bit level. Differences in observed values are sampling variation, not evidence of one source being cryptographically superior. CNG and OpenSSL are API wrappers over platform/implementation CSPRNG behavior; `/dev/urandom` exposes the Linux kernel interface. The implementation, entropy initialization, state handling, reseeding, and operational lifecycle determine security properties.

## Why statistical tests are insufficient
Statistical tests can reject obviously biased streams, but they cannot prove unpredictability. A generator could pass every sample test yet repeat a fixed stream or reveal future outputs if its state is recoverable. Passing statistics says only that observed data resembles random data under the selected tests. Cryptographic security additionally requires enough entropy, confidential state, forward/backtracking resistance, and resistance to state or output prediction.